<a href="https://colab.research.google.com/github/MinhLuan47/instagram-redirect/blob/main/Copy_Folder_Google_Drive_to_Google_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder Google Drive to Google Drive - 1TouchPro

In [1]:
#@title Input
from ipywidgets import widgets

dest_text = widgets.Text(description="Your drive", placeholder='Nhập đường link folder Google Drive của bạn')
source_text = widgets.Text(description="Shared drive", placeholder='Nhập đường link folder Google Drive shared')
from_page_text = widgets.Text(description="Từ trang", value="0")
to_page_text = widgets.Text(description="Đến trang", value="0")
max_download_size_text = widgets.Text(description="Tổng dung lượng tối đa(GB)", value="700")
exclude_str_text = widgets.Text(description="Bỏ file, folder có chứa nội dung", value="")

display(dest_text)
display(source_text)
display(from_page_text)
display(to_page_text)
display(max_download_size_text)
display(exclude_str_text)

Text(value='', description='Your drive', placeholder='Nhập đường link folder Google Drive của bạn')

Text(value='', description='Shared drive', placeholder='Nhập đường link folder Google Drive shared')

Text(value='0', description='Từ trang')

Text(value='0', description='Đến trang')

Text(value='700', description='Tổng dung lượng tối đa(GB)')

Text(value='', description='Bỏ file, folder có chứa nội dung')

In [9]:
#@title Run
import os
import time
import re
import sys
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import auth

class DriveCloner:
    def __init__(self):
        self._total_size = 0
        self._limit_size = 0
        self._count = 0
        self._total_files = 0
        self.excluded_strings = []

    # ===================== AUTH =====================
    def get_service(self):
        auth.authenticate_user()
        return build('drive', 'v3')

    # ===================== RETRY =====================
    def retry(self, func, max_retries=5):
        for i in range(max_retries):
            try:
                return func()
            except HttpError as e:
                wait = (2 ** i) + random.random()
                print(f"Retry {i+1} | wait {wait:.2f}s | error: {e}")
                time.sleep(wait)
        raise Exception("Max retries exceeded")

    # ===================== EXTRACT ID =====================
    def extract_id(self, url):
        match = re.search(r'[-\w]{25,}', url)
        return match.group(0) if match else None

    # ===================== LIST FILES =====================
    def list_files(self, service, folder_id):
        files = []
        page_token = None

        query = f"'{folder_id}' in parents and trashed=false"

        if self.excluded_strings:
            exclude_query = " and ".join(
                [f"not name contains '{s}'" for s in self.excluded_strings]
            )
            query += f" and ({exclude_query})"

        while True:
            response = self.retry(lambda: service.files().list(
                q=query,
                fields="files(id,name,mimeType,size,shortcutDetails),nextPageToken",
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
                pageToken=page_token
            ).execute())

            files.extend(response.get('files', []))
            page_token = response.get('nextPageToken')

            if not page_token:
                break

        return files

    # ===================== CHECK EXIST =====================
    def exists(self, service, parent_id, name):
        try:
            name = name.replace("'", "\\'")
            res = service.files().list(
                q=f"'{parent_id}' in parents and name = '{name}' and trashed=false",
                fields="files(id)",
                supportsAllDrives=True
            ).execute()

            return res['files'][0]['id'] if res.get('files') else None
        except:
            return None

    # ===================== CREATE FOLDER =====================
    def create_folder(self, service, parent_id, name):
        exist = self.exists(service, parent_id, name)
        if exist:
            return exist

        folder = self.retry(lambda: service.files().create(
            body={
                "name": name,
                "mimeType": "application/vnd.google-apps.folder",
                "parents": [parent_id]
            },
            fields="id",
            supportsAllDrives=True
        ).execute())

        return folder['id']

    # ===================== COPY FILE =====================
    def copy_file(self, service, parent_id, file):
        try:
            # handle shortcut
            file_id = file['id']
            if file.get('shortcutDetails'):
                file_id = file['shortcutDetails']['targetId']

            # folder
            if file['mimeType'] == 'application/vnd.google-apps.folder':
                new_folder_id = self.create_folder(service, parent_id, file['name'])
                children = self.list_files(service, file_id)
                self.copy_batch(service, new_folder_id, children)
                return

            # file
            if self.exists(service, parent_id, file['name']):
                print(f"Skip (exists): {file['name']}")
                return

            start = time.time()

            self.retry(lambda: service.files().copy(
                fileId=file_id,
                body={"parents": [parent_id]},
                supportsAllDrives=True
            ).execute())

            end = time.time()

            size_mb = int(file.get('size', 0)) / (1024 * 1024)
            self._total_size += size_mb
            self._count += 1

            speed = size_mb / (end - start) if (end - start) > 0 else 0

            print(f"[{self._count}/{self._total_files}] {file['name']} | {size_mb:.2f} MB | {speed:.2f} MB/s")

            # limit size
            if self._total_size >= self._limit_size * 1024:
                print("Reached max size limit")
                sys.exit()

        except Exception as e:
            print(f"Error file {file['name']}: {e}")

    # ===================== MULTI THREAD =====================
    def copy_batch(self, service, parent_id, files):
        with ThreadPoolExecutor(max_workers=5) as executor:
            futures = [
                executor.submit(self.copy_file, service, parent_id, f)
                for f in files
            ]
            for _ in as_completed(futures):
                pass

    # ===================== MAIN =====================
    def clone(self, dest_link, source_link):
        service = self.get_service()

        dest_id = self.extract_id(dest_link)
        source_id = self.extract_id(source_link)

        source_meta = service.files().get(
            fileId=source_id,
            supportsAllDrives=True
        ).execute()

        root_folder = self.create_folder(service, dest_id, source_meta['name'])

        files = self.list_files(service, source_id)
        self._total_files = len(files)

        print(f"Total files: {self._total_files}")

        start = time.time()
        self.copy_batch(service, root_folder, files)
        end = time.time()

        print("\nDONE")
        print(f"Total size: {self._total_size/1024:.2f} GB")
        print(f"Time: {int(end-start)}s")

        # RUN
cloner = DriveCloner()
cloner._limit_size = float(max_download_size_text.value)
cloner.excluded_strings = [s.strip() for s in exclude_str_text.value.split(",") if s.strip()]

cloner.clone(dest_text.value, source_text.value)

HttpError: <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files/1BRSNJOcYq1L1nbK-iNklQ9OPt-vD6otx?supportsAllDrives=true&alt=json returned "File not found: 1BRSNJOcYq1L1nbK-iNklQ9OPt-vD6otx.". Details: "[{'message': 'File not found: 1BRSNJOcYq1L1nbK-iNklQ9OPt-vD6otx.', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">